# Fig. 7: CDFs of the released algorithms

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig02"]
seed_everything(cfg["seed"])
record_run("Fig02", cfg)


In [ ]:
with torch.no_grad():
    for I_max in cfg["classical_iterations"]:
        # This script evaluates the classical WMMSE algorithm on the test probInsts dataset generated from DeepMIMO channels

        import torch
        import numpy as np
        import matplotlib.pyplot as plt
        from tqdm import tqdm
        from solutions.trihybrid_WMMSE import TriHybridWMMSE
        from solutions.utils import computeWSR, HybridPrecodingBFGS, checkHybridConstraints
        import os

        cdtype = torch.cfloat

        SAVE_DIR = DATA_DIR
        data = load_artifact(os.path.join(SAVE_DIR, "probInsts_test.pt"))
        parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
        ProbInsts = data["ProbInsts"][:cfg["max_instances"]]
        Nt = parameters["Nt"]
        Nc = parameters["Nc"]
        BW = parameters["BW"]
        P = parameters["P"]
        Mk = parameters["Mk"]
        eta = parameters["eta"]
        Ns_k = parameters["Ns_k"]
        sigma2 = parameters["sigma2"]

        # =====================================================
        # Run all problem instances & collect sum-rates
        # =====================================================
        sum_rates = []
        for pro_ind in tqdm(range(len(ProbInsts)), desc="Running WMMSE on all instances"):

            Inst = ProbInsts[pro_ind]
            H_list = Inst["H"]
            K = Inst["K"]
            Ns = Ns_k * K
            # Initialize u0 and v0
            u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
            v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
            for n in range(Nt):    # normalize v0
                norm_vn = torch.linalg.norm(v0[n])
                if norm_vn > torch.sqrt(P):
                    v0[n] = v0[n] / norm_vn * torch.sqrt(P)
            # Solver
            solver = TriHybridWMMSE(
                H_list=H_list,
                Ns_k=[Ns_k] * K,
                Mk=[Mk] * K,
                sigma2=[sigma2] * K,
                beta=[1.0] * K,
                P=[P] * Nt,
                eta=eta,
                u_init=u0,
                v_init=v0,
            )
            # Run classical WMMSE
            rate_trace, FA_opt, FD_opt, _, _ = solver.run_classical(I_max=I_max, verbose=False)

            # Decompose fully-digital precoder into hybrid components
            Nrf = cfg["rf_chains"]
            solver = HybridPrecodingBFGS(FD_opt, Nrf)
            FRF_opt, FBB_opt = solver.solve(max_iter=100, use_B0=True, verbose=False)
            phase_ok, papc_ok, info = checkHybridConstraints(FRF_opt, FBB_opt, P, tol=1e-6, verbose=False)
            if not phase_ok:
                print("Warning: Phase shifter constraint violated!")
            if not papc_ok:
                FBB_opt = FBB_opt / torch.sqrt(torch.tensor(info["max_papc_ratio"]))   
            R_final, _ = computeWSR(FA_opt, FRF_opt, FBB_opt, H_list, parameters)

            sum_rates.append(R_final.item())

        # =====================================================
        # Plot histogram of sum-rates
        # =====================================================
        sum_rates = np.array(sum_rates)
        plt.figure(figsize=(8, 4))
        plt.hist(sum_rates, bins=40, edgecolor='black')
        plt.xlabel("Optimized Sum-rate")
        plt.ylabel("Count")
        plt.title("Distribution of Sum-rate over Problem Instances")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        mean_rate = np.mean(sum_rates)
        var_rate = np.var(sum_rates)
        print(f"Mean sum-rate: {mean_rate:.4f}")
        print(f"Variance of sum-rate: {var_rate:.4f}")

        # =====================================================
        # Compute CDF of sum-rates
        # =====================================================
        sum_rates_sorted = np.sort(sum_rates)
        cdf = np.arange(1, len(sum_rates_sorted) + 1) / len(sum_rates_sorted)

        plt.figure(figsize=(6, 4))
        plt.plot(sum_rates_sorted, cdf, linewidth=2)
        plt.xlabel("Sum-rate")
        plt.ylabel("CDF")
        plt.title("CDF of Optimized Sum-rate (Classical WMMSE)")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # =====================================================
        # Save CDF data
        # =====================================================
        np.savez(
            os.path.join(result_dir("results_CDF"), f"CDF_sumrate_WMMSE_Imax_{I_max}.npz"),
            sum_rates=sum_rates,
            sum_rates_sorted=sum_rates_sorted,
            cdf=cdf,
            mean_rate=mean_rate,
            var_rate=var_rate,
            I_max=I_max
        )

        print("[Info] CDF data saved")


In [ ]:
with torch.no_grad():
    for I_max in cfg["unfolded_layers"]:
        # This script evaluates the unfolded WMMSE algorithm on the test dataset generated from DeepMIMO channels

        import torch
        import numpy as np
        import matplotlib.pyplot as plt
        from tqdm import tqdm
        from solutions.trihybrid_WMMSE import TriHybridWMMSE, GNN
        from solutions.utils import computeWSR, channelDataset, HybridPrecodingBFGS, checkHybridConstraints
        import os


        cdtype = torch.cfloat

        SAVE_DIR = DATA_DIR
        data = load_artifact(os.path.join(SAVE_DIR, "probInsts_test.pt"))
        parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
        ProbInsts = data["ProbInsts"][:cfg["max_instances"]]
        Nt = parameters["Nt"]
        Nc = parameters["Nc"]
        BW = parameters["BW"]
        P = parameters["P"]
        Mk = parameters["Mk"]
        eta = parameters["eta"]
        Ns_k = parameters["Ns_k"]
        sigma2 = parameters["sigma2"]

        # =====================================================
        # Define deep GNN model
        # =====================================================
        in_dim = 2 * ProbInsts[0]["H"][0].numel() + 1    # input feature dimension per node
        GNNs_gamma = torch.nn.ModuleList()
        GNNs_u = torch.nn.ModuleList()
        Ns_k = parameters["Ns_k"]
        for _ in range(I_max):
            GNN_gamma = GNN(
                in_dim=in_dim,
                hidden_dim=2*Ns_k**2+4,
                out_dim=2*Ns_k**2,
                num_layers=3,
                dp_rate=0.1
            )
            GNN_u = GNN(
                in_dim=in_dim,
                hidden_dim=Nt+10,
                out_dim=Nt,
                num_layers=3,
                dp_rate=0.1
            )
            GNNs_gamma.append(GNN_gamma)
            GNNs_u.append(GNN_u)

        # Load trained model
        ckpt_path = checkpoint_path(I_max)
        checkpoint = load_artifact(ckpt_path)
        for i in range(I_max):
            GNNs_gamma[i].load_state_dict(checkpoint["GNNs_gamma_state_dict"][i])
            GNNs_u[i].load_state_dict(checkpoint["GNNs_u_state_dict"][i])
        GNNs_gamma.eval()
        GNNs_u.eval()
        print("[Info] Loaded trained model from", ckpt_path)

        # =====================================================
        # Generate dataset
        # =====================================================
        dataset = channelDataset(ProbInsts, parameters)

        # =====================================================
        # Run all problem instances & collect sum-rates
        # =====================================================
        sum_rates = []
        for pro_ind in tqdm(range(len(ProbInsts)), desc="Running UWMMSE on all instances"):

            pro_inst = dataset.__getitem__(pro_ind)
            H_list = pro_inst["H_list"]
            K = pro_inst["K"]

            # System parameters per instance 
            Ns = Ns_k * K
            # Initialize u0 and v0
            u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
            v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
            for n in range(Nt):    # normalize v0
                norm_vn = torch.linalg.norm(v0[n])
                if norm_vn > torch.sqrt(P):
                    v0[n] = v0[n] / norm_vn * torch.sqrt(P)
    
            # --- Graph ---
            node_features = pro_inst["node_feature"]
            edge_index = pro_inst["edge_index"]

            # --- GNN output ---
            delta_gamma_list = []
            delta_u_list = []
            graph_id = torch.zeros(K, dtype=torch.int32)
            num_graphs = torch.tensor(1, dtype=torch.int32)
            for i in range(I_max):
                # delta_gamma
                delta_gamma = GNNs_gamma[i](node_features, edge_index)
                delta_gamma_list.append(delta_gamma)
                # delta_u
                delta_u = GNNs_u[i](node_features, edge_index, graph_id, num_graphs)
                delta_u_mt = delta_u.reshape(Nt, 1)
                delta_u_list.append(delta_u_mt)

            # Solver
            solver = TriHybridWMMSE(
                H_list=H_list,
                Ns_k=[Ns_k] * K,
                Mk=[Mk] * K,
                sigma2=[sigma2] * K,
                beta=[1.0] * K,
                P=[P] * Nt,
                eta=eta,
                u_init=u0,
                v_init=v0
            )
            # Run unfolded WMMSE
            rate_trace, FA_opt, FD_opt, _, _ = solver.run_unfolded(
                        dgamma_list=delta_gamma_list,
                        du_list=delta_u_list,
                        I_max=I_max,
                        verbose=False
                    )
            # Decompose fully-digital precoder into hybrid components
            Nrf = cfg["rf_chains"]
            solver = HybridPrecodingBFGS(FD_opt, Nrf)
            FRF_opt, FBB_opt = solver.solve(max_iter=100, use_B0=True, verbose=False)
            phase_ok, papc_ok, info = checkHybridConstraints(FRF_opt, FBB_opt, P, tol=1e-6, verbose=False)
            if not phase_ok:
                print("Warning: Phase shifter constraint violated!")
            if not papc_ok:
                FBB_opt = FBB_opt / torch.sqrt(torch.tensor(info["max_papc_ratio"]))   
            R_final, _ = computeWSR(FA_opt, FRF_opt, FBB_opt, H_list, parameters)
            # Record performance
            sum_rates.append(R_final.item())

        # Convert to numpy for convenience
        sum_rates = np.array(sum_rates)

        # =====================================================
        # Plot histogram of sum-rates
        # =====================================================
        plt.figure(figsize=(8, 4))
        plt.hist(sum_rates, bins=40, edgecolor='black')
        plt.xlabel("Optimized Sum-rate")
        plt.ylabel("Count")
        plt.title("Distribution of Sum-rate over Problem Instances")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        mean_rate = np.mean(sum_rates)
        var_rate = np.var(sum_rates)

        print(f"Mean sum-rate: {mean_rate:.4f}")
        print(f"Variance of sum-rate: {var_rate:.4f}")

        # =====================================================
        # Compute CDF of sum-rates
        # =====================================================
        sum_rates_sorted = np.sort(sum_rates)
        cdf = np.arange(1, len(sum_rates_sorted) + 1) / len(sum_rates_sorted)

        plt.figure(figsize=(6, 4))
        plt.plot(sum_rates_sorted, cdf, linewidth=2)
        plt.xlabel("Sum-rate")
        plt.ylabel("CDF")
        plt.title("CDF of Optimized Sum-rate (Unfolded WMMSE)")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # =====================================================
        # Save CDF data
        # =====================================================
        np.savez(
            os.path.join(result_dir("results_CDF"), f"CDF_sumrate_UWMMSE_Imax_{I_max}.npz"),
            sum_rates=sum_rates,
            sum_rates_sorted=sum_rates_sorted,
            cdf=cdf,
            mean_rate=mean_rate,
            var_rate=var_rate,
            I_max=I_max
        )

        print("[Info] CDF data saved")


In [ ]:
import os
# This script compares CDFs of sum-rates achieved by different algorithms on the DeepMIMO test dataset

import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))

# CDF proposed UWMMSE, Iter=2
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_2.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, marker='>',
    markevery=150, color='red', label="Unfolded algorithm, Iter=2", zorder=10)

# CDF proposed UWMMSE, Iter=3
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_3.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, marker='o',
    markevery=150, color='red', label="Unfolded algorithm, Iter=3", zorder=10)

# CDF proposed UWMMSE, Iter=4
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_4.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, marker='s',
    markevery=150, color='red', label="Unfolded algorithm, Iter=4", zorder=10)


# CDF 1
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_4.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, color ='#7F7F7F', label="Model-based algo, Iter=4")

# CDF 2
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_10.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, color ='#1F77B4', label="Model-based algo, Iter=10")

# CDF 3
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_50.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, color ='#2CA02C', label="Model-based algo, Iter=50")


# CDF 4
data = np.load(os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_200.npz"))
sum_rates_sorted = data["sum_rates_sorted"]   # x 
cdf = data["cdf"]                             # y 
plt.plot(sum_rates_sorted, cdf, linewidth=2, color ='black', label="Model-based algo, Iter=200")

plt.legend()
plt.xlabel("Sum-Rate")
plt.ylabel("CDF")
plt.title("CDF of Optimized Sum-rate in Dataset of Miami City")
plt.grid(True)
plt.tight_layout()



In [ ]:
import os
import numpy as np

files = [
    ("Model-based algo, Iter=4", os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_4.npz"), "black"),
    ("Model-based algo, Iter=10", os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_10.npz"), "blue"),
    ("Model-based algo, Iter=50", os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_50.npz"), "green"),
    ("Model-based algo, Iter=200", os.path.join(result_dir("results_CDF"), "CDF_sumrate_WMMSE_Imax_200.npz"), "gray"),
    ("Unfolded algo, Iter=2", os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_2.npz"), "red"),
    ("Unfolded algo, Iter=3", os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_3.npz"), "red"),
    ("Unfolded algo, Iter=4", os.path.join(result_dir("results_CDF"), "CDF_sumrate_UWMMSE_Imax_4.npz"), "red"),
]

for label, path, color in files:
    data = np.load(path)
    x = data["sum_rates_sorted"]
    y = data["cdf"]

    print(r"\addplot[")
    print(r"    line width=1pt,")
    print(f"    color={color},")
    print(r"]table[row sep=crcr]{%")

    for xi, yi in zip(x, y):
        print(f"{xi}\t{yi}\\\\")
    print("};")
    print(f"\\addlegendentry{{{label}}}")
    print("\n")